# 01 Data Loading and Cleaning

**Project:** STAT GR5243 Project 4: End-to-End Machine Learning  
**Topic:** Predicting High-Occupancy Airbnb Listings in New York City

This notebook loads the raw Inside Airbnb NYC data, checks basic data quality issues, creates the target variable, cleans selected predictors, and saves a processed dataset for EDA and modeling.


## 1. Import Libraries and Define Paths

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

raw_path = Path("../data/raw")
processed_path = Path("../data/processed")
processed_path.mkdir(parents=True, exist_ok=True)


## 2. Load Raw Data

In [ ]:
# Load detailed listings data
listings = pd.read_csv(raw_path / "listings.csv.gz", compression="gzip", low_memory=False)

# Load summary listings data
summary = pd.read_csv(raw_path / "listings.csv", low_memory=False)

# Load neighbourhood data
neighbourhoods = pd.read_csv(raw_path / "neighbourhoods.csv")

print("Detailed listings shape:", listings.shape)
print("Summary listings shape:", summary.shape)
print("Neighbourhoods shape:", neighbourhoods.shape)


## 3. Initial Data Quality Check

The original idea was to predict listing price. However, in the downloaded NYC data, the `price` column is fully missing in both the detailed and summary listings files. Therefore, we shift the project to predicting high-occupancy listings using `estimated_occupancy_l365d`, which is available and complete.


In [ ]:
# Check price availability
print("Detailed price missing rate:", listings["price"].isna().mean())
print("Summary price missing rate:", summary["price"].isna().mean())

# Check occupancy variable
print(listings["estimated_occupancy_l365d"].describe())
print(listings["estimated_occupancy_l365d"].value_counts(dropna=False).head(10))


## 4. Define Modeling Target

We define a binary classification target:

- `high_occupancy = 1` if `estimated_occupancy_l365d >= 60`
- `high_occupancy = 0` otherwise

The value 60 corresponds roughly to the upper quartile of the occupancy distribution, which separates more active listings from lower-activity listings.


In [ ]:
# Create target variable
listings["high_occupancy"] = (listings["estimated_occupancy_l365d"] >= 60).astype(int)

print(listings["high_occupancy"].value_counts())
print(listings["high_occupancy"].value_counts(normalize=True))


## 5. Select Useful Variables

We select variables related to host characteristics, location, property type, room type, availability, reviews, and listing capacity.


In [ ]:
selected_cols = [
    "id",
    "listing_url",
    "host_since",
    "hosts_time_as_host_years",
    "host_response_time",
    "host_response_rate",
    "host_acceptance_rate",
    "host_is_superhost",
    "host_listings_count",
    "host_total_listings_count",
    "host_identity_verified",
    "neighbourhood_cleansed",
    "neighbourhood_group_cleansed",
    "latitude",
    "longitude",
    "property_type",
    "room_type",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "minimum_nights",
    "maximum_nights",
    "availability_30",
    "availability_60",
    "availability_90",
    "availability_365",
    "availability_eoy",
    "number_of_reviews",
    "number_of_reviews_ltm",
    "number_of_reviews_l30d",
    "number_of_reviews_ly",
    "reviews_per_month",
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value",
    "instant_bookable",
    "calculated_host_listings_count",
    "calculated_host_listings_count_entire_homes",
    "calculated_host_listings_count_private_rooms",
    "calculated_host_listings_count_shared_rooms",
    "estimated_occupancy_l365d",
    "high_occupancy"
]

available_cols = [col for col in selected_cols if col in listings.columns]
df = listings[available_cols].copy()

print("Selected data shape:", df.shape)
df.head()


## 6. Clean Variables

In [ ]:
# Convert percentage strings to numeric values
for col in ["host_response_rate", "host_acceptance_rate"]:
    if col in df.columns:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace("%", "", regex=False)
            .replace({"nan": np.nan, "None": np.nan})
            .astype(float)
        )

# Convert true/false variables to binary indicators
binary_cols = ["host_is_superhost", "host_identity_verified", "instant_bookable"]

for col in binary_cols:
    if col in df.columns:
        df[col] = df[col].map({"t": 1, "f": 0})

# Fill missing review frequency with 0
if "reviews_per_month" in df.columns:
    df["reviews_per_month"] = df["reviews_per_month"].fillna(0)

# Add missing indicators for review score variables, then median-impute
review_cols = [
    "review_scores_rating",
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value"
]

for col in review_cols:
    if col in df.columns:
        df[col + "_missing"] = df[col].isna().astype(int)
        median_value = df[col].median()
        if pd.isna(median_value):
            median_value = 0
        df[col] = df[col].fillna(median_value)

# Median-impute numeric variables; if a column is completely missing, fill with 0
numeric_cols = [
    "hosts_time_as_host_years",
    "host_response_rate",
    "host_acceptance_rate",
    "host_listings_count",
    "host_total_listings_count",
    "latitude",
    "longitude",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "minimum_nights",
    "maximum_nights",
    "availability_30",
    "availability_60",
    "availability_90",
    "availability_365",
    "availability_eoy",
    "number_of_reviews",
    "number_of_reviews_ltm",
    "number_of_reviews_l30d",
    "number_of_reviews_ly",
    "calculated_host_listings_count",
    "calculated_host_listings_count_entire_homes",
    "calculated_host_listings_count_private_rooms",
    "calculated_host_listings_count_shared_rooms"
]

for col in numeric_cols:
    if col in df.columns:
        median_value = df[col].median()
        if pd.isna(median_value):
            median_value = 0
        df[col] = df[col].fillna(median_value)

# Fill binary variables with mode; if no mode exists, fill with 0
for col in binary_cols:
    if col in df.columns:
        modes = df[col].dropna().mode()
        fill_value = modes.iloc[0] if len(modes) > 0 else 0
        df[col] = df[col].fillna(fill_value)

# Fill categorical variables
categorical_cols = [
    "host_response_time",
    "neighbourhood_cleansed",
    "neighbourhood_group_cleansed",
    "property_type",
    "room_type"
]

for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

# Drop rows missing essential target/location fields
df = df.dropna(subset=["estimated_occupancy_l365d", "latitude", "longitude"])

print("Cleaned data shape:", df.shape)
print("Remaining missing values:", df.isna().sum().sum())


## 7. Save Processed Dataset

In [ ]:
output_file = processed_path / "nyc_airbnb_cleaned.csv"
df.to_csv(output_file, index=False)

print("Saved cleaned dataset to:", output_file)
print(df.shape)
df.head()


## 8. Cleaning Summary

Key cleaning decisions:

1. Price prediction was not used because the `price` column is fully missing in both detailed and summary listing files.
2. The target variable `high_occupancy` was created from `estimated_occupancy_l365d`.
3. Percentage columns such as host response rate and acceptance rate were converted to numeric values.
4. Binary true/false variables were converted to 0/1 indicators.
5. Missing review scores were median-imputed, with additional missing-value indicators.
6. Missing numerical values were imputed using medians; columns that were completely missing were filled with 0.
7. Missing categorical values were labeled as `Unknown`.

The cleaned dataset can now be used for EDA, unsupervised learning, feature engineering, and supervised classification modeling.
